# **Dataset for Doamin Adapter**

In [ ]:
!pip install wikipedia-api -q


In [ ]:
import wikipediaapi
import json
import re
import random

# Wikipedia REQUIRES a descriptive User-Agent identifying your project.
# Replace the email with your own (or leave as-is, it still works).
wiki = wikipediaapi.Wikipedia(
    user_agent="Day30DomainAdaptation/1.0 ([email protected])",
    language="en"
)

In [ ]:
TOPICS = [
    "Deepfake",
    "Synthetic media",
    "AI safety",
    "Misinformation",
    "Disinformation",
    "AI alignment",
    "Content authenticity",
    "Coalition for Content Provenance and Authenticity",
    "Artificial intelligence regulation",
    "Election misinformation",
    "Deepfake pornography",
    "Fake news",
    "Generative artificial intelligence",
    "Large language model",
    "Prompt injection",
    "AI-generated media",
    "Digital watermarking",
    "Media literacy",
    "Facial recognition system",
    "Voice cloning",
    "Text-to-image model",
    "OpenAI",
    "EU Artificial Intelligence Act",
    "Algorithmic bias",
    "Hallucination (artificial intelligence)",
]

print(f"Total topics: {len(TOPICS)}")

Total topics: 25


In [ ]:
raw_articles = {}

for topic in TOPICS:
    try:
        page = wiki.page(topic)
        if page.exists():
            raw_articles[topic] = page.text
            print(f"[OK]   {topic}  ({len(page.text)} chars)")
        else:
            print(f"[SKIP] {topic}  (page does not exist)")
    except Exception as e:
        print(f"[SKIP] {topic}  ({e})")

print(f"\nSuccessfully fetched: {len(raw_articles)}/{len(TOPICS)} articles")

[OK]   Deepfake  (59962 chars)
[OK]   Synthetic media  (23816 chars)
[OK]   AI safety  (33210 chars)
[OK]   Misinformation  (61430 chars)
[OK]   Disinformation  (25464 chars)
[OK]   AI alignment  (43401 chars)
[SKIP] Content authenticity  (page does not exist)
[OK]   Coalition for Content Provenance and Authenticity  (5237 chars)
[OK]   Artificial intelligence regulation  (64963 chars)
[SKIP] Election misinformation  (page does not exist)
[OK]   Deepfake pornography  (17489 chars)
[OK]   Fake news  (131376 chars)
[OK]   Generative artificial intelligence  (36970 chars)
[OK]   Large language model  (53025 chars)
[OK]   Prompt injection  (11293 chars)
[OK]   AI-generated media  (23816 chars)
[OK]   Digital watermarking  (14067 chars)
[OK]   Media literacy  (33368 chars)
[OK]   Facial recognition system  (76853 chars)
[OK]   Voice cloning  (19382 chars)
[OK]   Text-to-image model  (7482 chars)
[OK]   OpenAI  (53253 chars)
[OK]   EU Artificial Intelligence Act  (22801 chars)
[OK]   Algorit

In [ ]:
def clean_text(text):
    # remove wikipedia section headers like "== History ==" and "=== Sub ==="
    text = re.sub(r"={2,}\s*.*?\s*={2,}", "", text)
    # remove citation-style brackets [1], [2], etc. (wikipedia lib usually strips these, but just in case)
    text = re.sub(r"\[\d+\]", "", text)
    # collapse multiple newlines/spaces
    text = re.sub(r"\n{2,}", "\n", text)
    text = re.sub(r" {2,}", " ", text)
    return text.strip()

def split_into_passages(text, min_sentences=3, max_sentences=6, min_chars=200):
    # simple sentence splitter (good enough for Wikipedia prose)
    sentences = re.split(r'(?<=[.!?])\s+', text)
    sentences = [s.strip() for s in sentences if len(s.strip()) > 15]

    passages = []
    i = 0
    while i < len(sentences):
        chunk_size = random.randint(min_sentences, max_sentences)
        chunk = " ".join(sentences[i:i + chunk_size])
        if len(chunk) >= min_chars:
            passages.append(chunk)
        i += chunk_size
    return passages

all_passages = []
for topic, content in raw_articles.items():
    cleaned = clean_text(content)
    passages = split_into_passages(cleaned)
    all_passages.extend(passages)

print(f"Total raw passages generated: {len(all_passages)}")

# ------------------------------------------------------------
# CELL 5: Filter & dedupe (quality control)
# ------------------------------------------------------------
seen = set()
final_passages = []

for p in all_passages:
    p_norm = p.strip()
    # skip too-short or duplicate passages
    if len(p_norm) < 200 or len(p_norm) > 1200:
        continue
    if p_norm in seen:
        continue
    seen.add(p_norm)
    final_passages.append(p_norm)

# cap at 500 max (shuffle first so we get variety across topics, not just first N)
random.seed(42)
random.shuffle(final_passages)
final_passages = final_passages[:700]

print(f"Final passage count after filtering/capping: {len(final_passages)}")


Total raw passages generated: 1274
Final passage count after filtering/capping: 700


In [ ]:
OUTPUT_PATH = "day30_domain_corpus.jsonl"

with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
    for passage in final_passages:
        record = {"text": passage}
        f.write(json.dumps(record, ensure_ascii=False) + "\n")

print(f"Saved {len(final_passages)} passages to {OUTPUT_PATH}")


Saved 700 passages to day30_domain_corpus.jsonl


In [ ]:
with open(OUTPUT_PATH, "r", encoding="utf-8") as f:
    lines = f.readlines()

print(f"\nTotal lines in file: {len(lines)}\n")
print("=== Sample passages ===\n")
for line in lines[:3]:
    obj = json.loads(line)
    print(obj["text"])
    print("-" * 60)


Total lines in file: 700

=== Sample passages ===

According to Craig McClain, over 66% of Facebook users obtain news from the site. This, in combination with increased political polarization and filter bubbles, led to a tendency for readers to mainly read headlines. Numerous individuals and news outlets have stated that fake news may have influenced the outcome of the 2016 American Presidential Election.
------------------------------------------------------------
It also launched OpenAI o1, an early reasoning model that was internally codenamed strawberry. Additionally, ChatGPT Pro—a $200/month subscription service offering unlimited o1 access and enhanced voice features—was introduced, and preliminary benchmark results for the upcoming OpenAI o3 models were shared. On January 23, 2025, OpenAI released Operator, an AI agent and tool for accessing websites to execute goals defined by users. The feature was only available to Pro users in the United States. OpenAI released deep researc